# Phase 1: Manual Data Collection Pipeline (Model L — Nepali)

This notebook implements the manual data collection pipeline for the Nepali Language Model (Model L).
All sources use raw Wikimedia XML dumps, Internet Archive OCR, web scraping, or GitHub mirrors.
**No Hugging Face datasets used here.**

---

## Source Table (Final Audited Counts)

| File | Lines | Words | Source |
| :--- | :---: | :---: | :--- |
| `nepali_news_onlinekhabar_huge.jsonl` | 157,621 | 55,951,571 | Manual / Scraped |
| `nepali_wikipedia_manual.jsonl` | 28,825 | 9,354,153 | Manual / Scraped |
| `nepali_news_ekantipur.jsonl` | 40,003 | 9,285,552 | Manual / Scraped |
| `nepali_16news.jsonl` | 28,724 | 8,986,122 | Manual / Scraped |
| `nepali_ia_books.jsonl` | 58,182 | 6,139,661 | Manual / Scraped |
| `nepali_classical_books.jsonl` | 166,068 | 5,750,424 | Manual / Scraped |
| `nepali_magazine_corpus.jsonl` | 84,777 | 4,167,931 | Manual / Scraped |
| `nepali_news_setopati.jsonl` | 15,809 | 2,879,890 | Manual / Scraped |
| `nepali_news_manual.jsonl` | 6,591 | 2,851,849 | Manual / Scraped |
| `nepali_purnima_magazine.jsonl` | 43,932 | 1,665,219 | Manual / Scraped |
| `nepali_wiktionary.jsonl` | 12,083 | 1,462,945 | Manual / Scraped |
| `nepali_brihat_sabdakosh.jsonl` | 123,371 | 1,434,314 | Manual / Scraped |
| `nepali_bhagwat_mahapuran2.jsonl` | 52,268 | 1,406,629 | Manual / Scraped |
| `nepali_misc_books.jsonl` | 24,232 | 1,047,425 | Manual / Scraped |
| `nepali_all_vedas.jsonl` | 16,666 | 758,174 | Manual / Scraped |
| `nepali_skanda_purana.jsonl` | 13,405 | 681,169 | Manual / Scraped |
| `nepali_gorkhapatra.jsonl` | 15,249 | 593,828 | Manual / Scraped |
| `nepali_wikibooks.jsonl` | 206 | 502,239 | Manual / Scraped |
| `nepali_news_ratopati.jsonl` | 479 | 314,642 | Manual / Scraped |
| `nepali_news_onlinekhabar.jsonl` | 69 | 32,895 | Manual / Scraped |
| `sumnima_ocr.jsonl` | 6 | 423 | Manual / Scraped |
| **TOTAL** | **888,566** | **115,267,055 (115.27M)** | |

## 1. Imports & Setup

In [ ]:
import os, io, re, bz2, json, time, ssl, gzip, shutil, subprocess, threading, zipfile
import urllib.request, urllib.parse
import xml.etree.ElementTree as ET
from pathlib import Path
from queue import Queue

try:
    import requests, pymupdf, pytesseract
    from PIL import Image
    from tqdm import tqdm
except ImportError:
    pass

RAW_MANUAL_DIR = Path("../datasets/raw/manual/")
RAW_MANUAL_DIR.mkdir(parents=True, exist_ok=True)

_ctx = ssl._create_unverified_context()
_dev_re = re.compile(r'[\u0900-\u097F]')

def _ia_metadata(item_id):
    url = f"https://archive.org/metadata/{item_id}"
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, context=_ctx, timeout=20) as r:
        data = json.loads(r.read())
    return [(item_id, f["name"]) for f in data.get("files", []) if f.get("name", "").endswith(".txt")]

def _ia_fetch(item_id, filename):
    url = f"https://archive.org/download/{item_id}/{urllib.parse.quote(filename)}"
    for attempt in range(3):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, context=_ctx, timeout=45) as r:
                return r.read().decode("utf-8", errors="ignore")
        except Exception:
            time.sleep(2 ** attempt)
    return ""

def _ia_download_parallel(files, out_path, source_tag, n_threads=10):
    q = Queue()
    lock = threading.Lock()
    saved = [0]
    for item in files: q.put(item)
    def _worker():
        while not q.empty():
            try: item_id, fname = q.get_nowait()
            except: break
            content = _ia_fetch(item_id, fname)
            rows = [json.dumps({"text": p.strip(), "source": source_tag}, ensure_ascii=False)
                    for p in content.split("\n\n") if len(p.strip()) > 60 and _dev_re.search(p)]
            if rows:
                with lock:
                    with open(out_path, "a", encoding="utf-8") as fh: fh.write("\n".join(rows) + "\n")
                    saved[0] += len(rows)
            q.task_done()
    threads = [threading.Thread(target=_worker) for _ in range(n_threads)]
    for t in threads: t.start()
    for t in threads: t.join()
    return saved[0]

def _parse_wiki_xml(dump_compressed, output_file, source_tag, skip_prefixes):
    ns_prefix = ""
    with bz2.open(dump_compressed, "rt", encoding="utf-8") as f:
        first_line = f.readline()
        m = re.search(r'xmlns=\"([^\"]+)\"', first_line)
        if m: ns_prefix = f"{{{m.group(1)}}}"
    pages_saved = 0
    with open(output_file, "w", encoding="utf-8") as out:
        with bz2.open(dump_compressed, "rt", encoding="utf-8") as f:
            for event, elem in ET.iterparse(f, events=("end",)):
                if elem.tag == f"{ns_prefix}page":
                    title_el = elem.find(f"{ns_prefix}title")
                    ns_el = elem.find(f"{ns_prefix}ns")
                    rev_el = elem.find(f"{ns_prefix}revision")
                    title = title_el.text if title_el is not None else ""
                    ns = ns_el.text if ns_el is not None else ""
                    if ns == "0" and title and not any(title.startswith(p) for p in skip_prefixes):
                        text_el = rev_el.find(f"{ns_prefix}text") if rev_el is not None else None
                        text = text_el.text if text_el is not None else ""
                        if text and len(text) > 150:
                            text = re.sub(r'\{\{[^}]*\}\}', '', text)
                            text = re.sub(r'\[\[[^\]]*\]\]', '', text)
                            text = re.sub(r'<[^>]*>', '', text).strip()
                            if len(text) > 150:
                                out.write(json.dumps({"text": text, "title": title, "source": source_tag}, ensure_ascii=False) + "\n")
                                pages_saved += 1
                    elem.clear()
    return pages_saved

print("Setup complete. RAW_MANUAL_DIR =", RAW_MANUAL_DIR.resolve())

## 2. Nepali Wikipedia XML Dump

In [ ]:
output_file = RAW_MANUAL_DIR / "nepali_wikipedia_manual.jsonl"
dump_url = "https://dumps.wikimedia.org/newiki/latest/newiki-latest-pages-articles.xml.bz2"
dump_compressed = RAW_MANUAL_DIR / "newiki-latest-pages-articles.xml.bz2"
ne_skip = ["ढाँचा:", "श्रेणी:", "चित्र:", "विकिपिडिया:", "सहायता:", "प्रयोगकर्ता:"]

if output_file.exists() and output_file.stat().st_size > 0:
    print(f"{output_file.name} already exists. Skipping.")
else:
    if not dump_compressed.exists():
        print("Downloading Nepali Wikipedia dump (~150 MB)...")
        req = urllib.request.Request(dump_url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as r, open(dump_compressed, "wb") as f:
            shutil.copyfileobj(r, f)
    n = _parse_wiki_xml(dump_compressed, output_file, "nepali_wikipedia_manual_dump", ne_skip)
    print(f"Done. {n:,} articles → {output_file.name}")

## 3. Nepali Wikibooks & Wiktionary

In [ ]:
for name, url_part in [("wikibooks", "newikibooks"), ("wiktionary", "newiktionary")]:
    output_file = RAW_MANUAL_DIR / f"nepali_{name}.jsonl"
    dump_url = f"https://dumps.wikimedia.org/{url_part}/latest/{url_part}-latest-pages-articles.xml.bz2"
    dump_compressed = RAW_MANUAL_DIR / f"{url_part}-latest-pages-articles.xml.bz2"
    ne_skip = ["ढाँचा:", "श्रेणी:", "चित्र:", "विकिपिडिया:", "सहायता:", "प्रयोगकर्ता:"]

    if output_file.exists() and output_file.stat().st_size > 0:
        print(f"{output_file.name} already exists. Skipping.")
        continue
    if not dump_compressed.exists():
        print(f"Downloading {name} dump...")
        req = urllib.request.Request(dump_url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as r, open(dump_compressed, "wb") as f:
            shutil.copyfileobj(r, f)
    n = _parse_wiki_xml(dump_compressed, output_file, f"nepali_{name}_dump", ne_skip)
    dump_compressed.unlink(missing_ok=True)
    print(f"Done. {n:,} pages → {output_file.name}")

## 4. Internet Archive: Classical Books, Vedas, Puranas, Magazines, Dictionary

In [ ]:
# All Internet Archive item collections
ia_sources = [
    ("nepali_classical_books.jsonl", [
        ("bhagwat-purana-in-nepali", "nepali_bhagwat_purana"),
        ("agni-purana-nepali", "nepali_agni_purana"),
        ("ShrimadbhagavadgitanepaliNarayan", "nepali_bhagavad_gita"),
        ("mulukiain1910", "nepali_muluki_ain_1910"),
        ("NepaliAinSangraha", "nepali_ain_sangraha"),
    ]),
    ("nepali_brihat_sabdakosh.jsonl", [
        ("NepaliDictionary-PragyaNEpaliBrihatSabdakosh", "nepali_brihat_sabdakosh"),
    ]),
    ("nepali_magazine_corpus.jsonl", [
        ("nepal-magazine", "nepali_magazine_nepal"),
        ("nari-magazine-collection", "nepali_magazine_nari"),
        ("fagun-magazine", "nepali_magazine_fagun"),
        ("swasthya-khabar", "nepali_magazine_swasthya_khabar"),
    ]),
    ("nepali_all_vedas.jsonl", [("all-ved-nepali", "nepali_all_vedas")]),
    ("nepali_purnima_magazine.jsonl", [("purnima-04-04_202407", "nepali_purnima_magazine")]),
    ("nepali_bhagwat_mahapuran2.jsonl", [("bhagwat-mahapuran-in-nepali_202203", "nepali_bhagwat_mahapuran")]),
    ("nepali_skanda_purana.jsonl", [("20230920_20230920_0856", "nepali_skanda_purana")]),
    ("nepali_gorkhapatra.jsonl", [("gorkhapatra-2019-10-13", "nepali_gorkhapatra_newspaper")]),
    ("nepali_misc_books.jsonl", [
        ("MaharshiDayanandaSatyarthaprakashNepaliNarayanPaudel", "nepali_satyarth_prakash"),
        ("20200712_20200712_1151", "nepali_nirnaya_sindhu"),
        ("Swasthani", "nepali_swasthani"),
        ("NepaliDaaktarNabhayemaaMahilaaKoLaagi", "nepali_medical_books"),
    ]),
]

for out_name, items in ia_sources:
    out = RAW_MANUAL_DIR / out_name
    if out.exists() and out.stat().st_size > 0:
        print(f"Already exists: {out.name}. Skipping.")
        continue
    for item_id, tag in items:
        try:
            files = _ia_metadata(item_id)
            n = _ia_download_parallel(files, out, tag, n_threads=8)
            print(f"  {item_id}: {n:,} paragraphs")
        except Exception as e:
            print(f"  {item_id}: FAILED ({e})")
    print(f"Done → {out.name}")

## 5. GitHub: 16NepaliNews & Nagarik/Setopati

In [ ]:
# --- 16NepaliNews (RAR from GitHub, needs `unar`) ---
out = RAW_MANUAL_DIR / "nepali_16news.jsonl"
if out.exists() and out.stat().st_size > 0:
    print(f"Already exists: {out.name}. Skipping.")
else:
    rar_url = "https://github.com/sndsabin/Nepali-News-Classifier/raw/master/16NepaliNews.rar"
    rar_path = RAW_MANUAL_DIR / "16NepaliNews.rar"
    extract_dir = RAW_MANUAL_DIR / "16NepaliNews_extract"
    print("Downloading 16NepaliNews.rar...")
    req = urllib.request.Request(rar_url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as r, open(rar_path, "wb") as f: f.write(r.read())
    extract_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(["/opt/homebrew/bin/unar", "-o", str(extract_dir), str(rar_path)], capture_output=True)
    saved = 0
    with open(out, "w", encoding="utf-8") as fh:
        for txt in (extract_dir / "16NepaliNews").glob("**/*.txt"):
            text = txt.read_text(encoding="utf-8", errors="ignore").strip()
            if len(text) > 80:
                fh.write(json.dumps({"text": text, "source": f"16nepali_news_{txt.parent.name.lower()}"}, ensure_ascii=False) + "\n")
                saved += 1
    rar_path.unlink(missing_ok=True); shutil.rmtree(extract_dir, ignore_errors=True)
    print(f"Done. {saved:,} articles → {out.name}")

# --- Nagarik & Setopati (ZIP from GitHub) ---
out = RAW_MANUAL_DIR / "nepali_news_manual.jsonl"
if out.exists() and out.stat().st_size > 0:
    print(f"Already exists: {out.name}. Skipping.")
else:
    zip_url = "https://github.com/ashmitbhattarai/Nepali-Language-Model-For-Next-Sentence-Prediction-Using-LSTM/archive/refs/heads/master.zip"
    zip_path = RAW_MANUAL_DIR / "nagarik_setopati.zip"
    extract = RAW_MANUAL_DIR / "nagarik_setopati_extract"
    print("Downloading Nagarik/Setopati ZIP...")
    req = urllib.request.Request(zip_url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as r, open(zip_path, "wb") as f: f.write(r.read())
    extract.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf: zf.extractall(extract)
    saved = 0
    with open(out, "w", encoding="utf-8") as fh:
        for txt in extract.glob("**/*.txt"):
            text = txt.read_text(encoding="utf-8", errors="ignore").strip()
            if len(text) > 80:
                fh.write(json.dumps({"text": text, "source": "nepali_news_nagarik_setopati"}, ensure_ascii=False) + "\n")
                saved += 1
    zip_path.unlink(missing_ok=True); shutil.rmtree(extract, ignore_errors=True)
    print(f"Done. {saved:,} articles → {out.name}")

## 6. Internet Archive: Bulk OCR Books

Searches IA for all Nepali-language texts with `_djvu.txt` OCR, downloads in parallel.

In [ ]:
def _ia_get(url, retries=3):
    for attempt in range(retries):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, context=_ctx, timeout=30) as r: return r.read()
        except: time.sleep(2 ** attempt)
    return b""

out_path = RAW_MANUAL_DIR / "nepali_ia_books.jsonl"
done_path = RAW_MANUAL_DIR / "nepali_ia_books.done_ids"

if out_path.exists() and out_path.stat().st_size > 50_000_000:
    print(f"Already exists: {out_path.name} ({out_path.stat().st_size // 1024 // 1024} MB). Skipping.")
else:
    done = set(done_path.read_text(encoding="utf-8").splitlines()) if done_path.exists() else set()
    # Search for Nepali books
    ids = []
    page = 1
    while len(ids) < 2000:
        url = f"https://archive.org/advancedsearch.php?q=language%3ANepali+AND+mediatype%3Atexts&fl[]=identifier&output=json&rows=200&page={page}"
        data = json.loads(_ia_get(url) or b"{}")
        docs = data.get("response", {}).get("docs", [])
        if not docs: break
        ids.extend(d["identifier"] for d in docs)
        page += 1; time.sleep(1)
    new_ids = [i for i in ids if i not in done]
    print(f"Items to check: {len(new_ids)}")

    # Get metadata
    meta_q, dl_q, ml = Queue(), Queue(), threading.Lock()
    for iid in new_ids: meta_q.put(iid)
    def _mw():
        while not meta_q.empty():
            try: iid = meta_q.get_nowait()
            except: break
            try:
                meta = json.loads(_ia_get(f"https://archive.org/metadata/{iid}") or b"{}")
                for f in meta.get("files", []):
                    if f.get("name", "").endswith("_djvu.txt"): dl_q.put((iid, f["name"])); break
                with ml: done.add(iid)
            except: pass
            time.sleep(0.3)
    ts = [threading.Thread(target=_mw) for _ in range(20)]
    for t in ts: t.start()
    for t in ts: t.join()
    print(f"Items with OCR: {dl_q.qsize()}")

    # Download
    lock, total = threading.Lock(), [0]
    _DEVA = re.compile(r"[\u0900-\u097F]")
    def _dw():
        while not dl_q.empty():
            try: iid, fname = dl_q.get_nowait()
            except: break
            content = (_ia_get(f"https://archive.org/download/{iid}/{urllib.parse.quote(fname)}") or b"").decode("utf-8", errors="ignore")
            rows = [json.dumps({"text": p.strip(), "source": "nepali_ia_books", "item_id": iid}, ensure_ascii=False)
                    for p in content.split("\n\n")
                    if len(p.split()) >= 40 and len(p.replace(' ','')) > 0
                    and len(_DEVA.findall(p)) / len(p.replace(' ','')) >= 0.35]
            if rows:
                with lock:
                    with open(out_path, "a", encoding="utf-8") as fh: fh.write("\n".join(rows) + "\n")
                    total[0] += len(rows)
            time.sleep(0.5)
    ts = [threading.Thread(target=_dw) for _ in range(12)]
    for t in ts: t.start()
    for t in ts: t.join()
    done_path.write_text("\n".join(done), encoding="utf-8")
    print(f"Done. {total[0]:,} paragraphs → {out_path.name}")

## 7. Local Tesseract OCR

Requires: `tesseract` with `nep` pack, `pymupdf`, `pytesseract`, `Pillow`.

In [ ]:
def process_pdf_ocr_nepali(pdf_url, pdf_filename, output_name):
    pdf_path = RAW_MANUAL_DIR / pdf_filename
    output_file = RAW_MANUAL_DIR / f"{output_name}_ocr.jsonl"
    if output_file.exists() and output_file.stat().st_size > 0:
        print(f"{output_file.name} already exists. Skipping.")
        return
    if not pdf_path.exists():
        print(f"Downloading {pdf_filename}...")
        r = requests.get(pdf_url, headers={"User-Agent": "Mozilla/5.0", "Referer": "https://archive.org"}, stream=True)
        if r.status_code != 200: print(f"Failed: HTTP {r.status_code}"); return
        with open(pdf_path, "wb") as f:
            for chunk in r.iter_content(65536): f.write(chunk)
    print(f"OCR on {pdf_path.name}...")
    doc = pymupdf.open(pdf_path)
    saved = 0
    with open(output_file, "w", encoding="utf-8") as fh:
        for pnum in tqdm(range(len(doc)), desc=f"OCR {output_name}"):
            pix = doc.load_page(pnum).get_pixmap(dpi=150)
            img = Image.open(io.BytesIO(pix.tobytes("png")))
            text = pytesseract.image_to_string(img, lang="nep").strip()
            if text and len(text) > 100:
                fh.write(json.dumps({"text": text, "page": pnum+1, "source": f"ocr_{output_name}"}, ensure_ascii=False) + "\n")
                saved += 1
    print(f"Done. {saved} pages → {output_file.name}")

process_pdf_ocr_nepali(
    "https://archive.org/download/SumnimaBiseshworP.Koirala/Sumnima%20%20Biseshwor%20P.%20Koirala.pdf",
    "sumnima.pdf", "sumnima")

## 8. News Site Scraping (Paginated)

Uses `scrape_news_paginated.py` (category pagination approach).
Scrapes ekantipur, setopati, ratopati, onlinekhabar.

**Run from terminal** (takes hours — rate-limited):
```bash
cd nepali/code && python3 scrape_news_paginated.py
```

Or run a specific site:
```bash
python3 scrape_news_paginated.py setopati
python3 scrape_news_paginated.py ekantipur
```

In [ ]:
# Quick check on existing news scraper output
for name in ["ekantipur", "setopati", "ratopati", "onlinekhabar"]:
    f = RAW_MANUAL_DIR / f"nepali_news_{name}.jsonl"
    if f.exists() and f.stat().st_size > 0:
        lines = sum(1 for _ in open(f))
        print(f"  {f.name}: {lines:,} articles, {f.stat().st_size//1024:,} KB")
    else:
        print(f"  {f.name}: not yet scraped")
print("\nTo scrape, run: python3 scrape_news_paginated.py")

## 9. Dataset Audit

## 18. Web Scrape: onlinekhabar.com (`nepali_news_onlinekhabar_huge.jsonl`)

Ultra-fast WordPress REST API scraper fetching 100 articles per JSON request using 12 concurrent connection-pooled threads.
Endpoint: `https://www.onlinekhabar.com/wp-json/wp/v2/posts?per_page=100&page={page}`
Decodes content explicitly as UTF-8 to preserve Devanagari script characters.


In [ ]:
import json, time, random, logging, sys, re
from pathlib import Path
from queue import Queue
import threading
import requests
from bs4 import BeautifulSoup
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

RAW_DIR = Path("../datasets/raw/manual")
RAW_DIR.mkdir(parents=True, exist_ok=True)

OUT_PATH  = RAW_DIR / "nepali_news_onlinekhabar_huge.jsonl"
SEEN_PATH = RAW_DIR / "nepali_news_onlinekhabar_huge.jsonl.seen"

DEVA_RE   = re.compile(r"[\u0900-\u097F]")
MIN_WORDS = 60
N_THREADS = 12

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept-Language": "ne,en;q=0.9",
}

def _is_nepali(text):
    chars = len(text.replace(" ", "").replace("\n", ""))
    if chars == 0: return False
    return len(DEVA_RE.findall(text)) / chars >= 0.40

def _clean_html(html):
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "nav", "footer", "header", "aside", "noscript"]):
        tag.decompose()
    return soup.get_text(separator="\n", strip=True)

def scrape_onlinekhabar_api(max_pages=2500, target_words=48_000_000):
    seen = set(SEEN_PATH.read_text(encoding="utf-8").splitlines()) if SEEN_PATH.exists() else set()
    print(f"Starting REST API scraper for onlinekhabar. Already seen {len(seen):,} IDs/URLs")

    page_q = Queue()
    lock = threading.Lock()
    saved = [0]
    words_saved = [0]
    stop_signal = [False]

    for p in range(1, max_pages + 1):
        page_q.put(p)

    def _api_worker():
        sess = requests.Session()
        sess.headers.update(HEADERS)
        while not page_q.empty() and not stop_signal[0]:
            try:
                page = page_q.get_nowait()
            except:
                break
            url = f"https://www.onlinekhabar.com/wp-json/wp/v2/posts?per_page=100&page={page}"
            try:
                r = sess.get(url, timeout=15, verify=False)
                if r.status_code in (400, 404):
                    page_q.task_done()
                    continue
                if r.status_code == 200:
                    posts = r.json()
                    if not posts:
                        page_q.task_done()
                        continue
                    batch_saved = batch_words = 0
                    lines_to_write, new_seen = [], []
                    for post in posts:
                        post_id = str(post.get("id", ""))
                        link = post.get("link", "")
                        if post_id in seen or link in seen:
                            continue
                        content_html = post.get("content", {}).get("rendered", "")
                        if not content_html:
                            continue
                        text = _clean_html(content_html)
                        w = len(text.split())
                        if w >= MIN_WORDS and _is_nepali(text):
                            item = {
                                "text": text,
                                "url": link,
                                "post_id": post_id,
                                "date": post.get("date", ""),
                                "source": "nepali_news_onlinekhabar_huge"
                            }
                            lines_to_write.append(json.dumps(item, ensure_ascii=False) + "\n")
                            new_seen.append(post_id)
                            if link: new_seen.append(link)
                            batch_saved += 1
                            batch_words += w
                    if lines_to_write:
                        with lock:
                            with open(OUT_PATH, "a", encoding="utf-8") as fh:
                                fh.writelines(lines_to_write)
                            seen.update(new_seen)
                            saved[0] += batch_saved
                            words_saved[0] += batch_words
                            if words_saved[0] >= target_words:
                                stop_signal[0] = True
            except Exception: pass
            page_q.task_done()

    threads = [threading.Thread(target=_api_worker, daemon=True) for _ in range(N_THREADS)]
    for t in threads: t.start()
    for t in threads: t.join()
    SEEN_PATH.write_text("\n".join(seen), encoding="utf-8")
    print(f"=== Complete! Saved {saved[0]:,} docs, {words_saved[0]/1e6:.2f}M words ===")

print("Scraper function defined.")


## 19. Web Scrape: ekantipur.com Date-Based Crawler (`nepali_news_ekantipur.jsonl`)

Date-based crawler for `ekantipur.com` — crawls news articles published on specific dates going backwards in time.

**Strategy**: Iterates day-by-day backwards from August 18, 2026, constructs date URLs (`/news/YYYY/MM/DD`),
gathers all article links listed on each date's index page, and downloads them in parallel.

- Resumable via `nepali_news_ekantipur.jsonl.seen` file
- Spawns 8 parallel download threads per day
- Filters to ≥40% Devanagari characters
- Robust to connection drops and SSL certification errors

In [ ]:
import re, json, time, random, threading, requests
from pathlib import Path
from datetime import datetime, timedelta
from queue import Queue
from bs4 import BeautifulSoup
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

_EK_RE = re.compile(r"[ऀ-ॿ]")
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept-Language": "ne,en;q=0.9",
}

def _ek_extract(html):
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "nav", "footer", "header", "aside", "noscript"]): tag.decompose()
    for sel in ["div.description", "div.news-desc", "div.article-content", "div[class*='detail']", "article"]:
        try:
            el = soup.select_one(sel)
            if el:
                t = el.get_text(separator="
", strip=True)
                if len(t.split()) >= 80: return t
        except: pass
    best, bn = "", 0
    for tag in soup.find_all(["article", "main", "div"]):
        t = tag.get_text(separator="
", strip=True)
        if len(t) > bn and len(t.split()) >= 80: best, bn = t, len(t)
    return best

def _ek_get_links(html):
    soup = BeautifulSoup(html, "html.parser")
    links = set()
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.startswith("/"): href = "https://ekantipur.com" + href
        if re.search(r"/news/\d{4}/\d{2}/\d{2}/", href):
            links.add(href.split("?")[0])
    return links

def run_ekantipur_dates(days_back=1200, n_threads=24):
    out_path  = RAW_MANUAL_DIR / "nepali_news_ekantipur.jsonl"
    seen_path = RAW_MANUAL_DIR / "nepali_news_ekantipur.jsonl.seen"
    seen = set(seen_path.read_text(encoding="utf-8").splitlines()) if seen_path.exists() else set()
    print(f"ekantipur: {len(seen):,} URLs already seen")

    start_date = datetime.strptime("2026-08-18", "%Y-%m-%d")
    dates = [start_date - timedelta(days=i) for i in range(days_back)]

    dl_q = Queue()
    lock = threading.Lock()
    saved, checked = [0], [0]

    def _dw():
        sess = requests.Session()
        sess.headers.update(HEADERS)
        while True:
            url = dl_q.get()
            if url is None: break
            try:
                r = sess.get(url, timeout=15, verify=False)
                if r.status_code == 200:
                    text = _ek_extract(r.text)
                    chars = len(text.replace(" ","").replace("
",""))
                    if chars > 0 and len(_EK_RE.findall(text)) / chars >= 0.40:
                        with lock:
                            with open(out_path, "a", encoding="utf-8") as fh:
                                fh.write(json.dumps({"text": text, "url": url,
                                    "source": "nepali_news_ekantipur"}, ensure_ascii=False) + "
")
                            saved[0] += 1
            except: pass
            dl_q.task_done()

    workers = [threading.Thread(target=_dw, daemon=True) for _ in range(n_threads)]
    for w in workers: w.start()

    main_sess = requests.Session()
    main_sess.headers.update(HEADERS)

    for dt in dates:
        dt_str = dt.strftime("%Y/%m/%d")
        url = f"https://ekantipur.com/news/{dt_str}"
        print(f"Fetching date listing: {dt_str}...")
        try:
            r = main_sess.get(url, timeout=15, verify=False)
            if r.status_code != 200: continue
            links = {u for u in _ek_get_links(r.text) if u not in seen}
            if not links: continue
            print(f"  Found {len(links)} new articles to download")
            for u in links:
                seen.add(u)
                dl_q.put(u)
            dl_q.join()
            checked[0] += len(links)
            with lock:
                seen_path.write_text("
".join(seen), encoding="utf-8")
                print(f"  Progress: Date {dt_str} | total saved={saved[0]:,}")
        except: pass

    for _ in range(n_threads): dl_q.put(None)
    for w in workers: w.join()
    print(f"ekantipur done: {saved[0]:,} articles")

run_ekantipur_dates(days_back=1200)


## 20. Web Scrape: setopati.com & ratopati.com (`nepali_news_setopati.jsonl` & `nepali_news_ratopati.jsonl`)

Category-based paginated crawlers for `setopati.com` and `ratopati.com` utilizing connection pooling and 16 concurrent threads.

- Setopati categories: social, politics, opinion, sports, art, global
- Ratopati categories: news, politics, economy, sports, entertainment, international, society, literature
- Delay: 0.1s delay
- Filters to ≥40% Devanagari characters

In [ ]:
import re, json, time, random, threading, requests
from bs4 import BeautifulSoup
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

_NE_RE = re.compile(r"[\u0900-\u097F]")
_SETO_CATS = [
    ("social",   "https://www.setopati.com/social?page={page}"),
    ("politics", "https://www.setopati.com/politics?page={page}"),
    ("opinion",  "https://www.setopati.com/opinion?page={page}"),
    ("sports",   "https://www.setopati.com/sports?page={page}"),
    ("art",      "https://www.setopati.com/art?page={page}"),
    ("global",   "https://www.setopati.com/global?page={page}"),
]
_RATO_CATS = [
    ("news",          "https://www.ratopati.com/category/news?page={page}"),
    ("politics",      "https://www.ratopati.com/category/politics?page={page}"),
    ("economy",       "https://www.ratopati.com/category/economy?page={page}"),
    ("sports",        "https://www.ratopati.com/category/sports?page={page}"),
    ("entertainment", "https://www.ratopati.com/category/entertainment?page={page}"),
    ("international", "https://www.ratopati.com/category/international?page={page}"),
    ("society",       "https://www.ratopati.com/category/society?page={page}"),
    ("literature",    "https://www.ratopati.com/category/literature?page={page}"),
]

def run_setopati_ratopati(n_threads=16):
    seto_out = RAW_MANUAL_DIR / "nepali_news_setopati.jsonl"
    seto_seen_path = RAW_MANUAL_DIR / "nepali_news_setopati.jsonl.seen"
    seto_seen = set(seto_seen_path.read_text(encoding="utf-8").splitlines()) if seto_seen_path.exists() else set()

    rato_out = RAW_MANUAL_DIR / "nepali_news_ratopati.jsonl"
    rato_seen_path = RAW_MANUAL_DIR / "nepali_news_ratopati.jsonl.seen"
    rato_seen = set(rato_seen_path.read_text(encoding="utf-8").splitlines()) if rato_seen_path.exists() else set()

    dl_q = Queue()
    lock = threading.Lock()
    saved = {"seto": 0, "rato": 0}

    def _dw():
        sess = requests.Session()
        sess.headers.update({"User-Agent": "Mozilla/5.0"})
        while True:
            item = dl_q.get()
            if item is None: break
            site, url = item
            try:
                r = sess.get(url, timeout=12, verify=False)
                if r.status_code == 200:
                    selectors = ["div.story-content", "div.article-content", "div.entry-content", "article"]
                    if site == "rato": selectors = ["div[class*='detail']", "div.story-content"] + selectors
                    
                    soup = BeautifulSoup(r.text, "html.parser")
                    for tag in soup(["script", "style", "nav", "footer", "header", "aside", "noscript"]): tag.decompose()
                    text = ""
                    for sel in selectors:
                        try:
                            el = soup.select_one(sel)
                            if el:
                                t = el.get_text(separator="\n", strip=True)
                                if len(t.split()) >= 80: text = t; break
                        except: pass
                    
                    chars = len(text.replace(" ","").replace("\n",""))
                    if chars > 0 and len(_NE_RE.findall(text)) / chars >= 0.40:
                        with lock:
                            out_f = seto_out if site == "seto" else rato_out
                            with open(out_f, "a", encoding="utf-8") as fh:
                                fh.write(json.dumps({"text": text, "url": url,
                                    "source": f"nepali_news_{site}pati"}, ensure_ascii=False) + "\n")
                            saved[site] += 1
            except: pass
            dl_q.task_done()

    workers = [threading.Thread(target=_dw, daemon=True) for _ in range(n_threads)]
    for w in workers: w.start()

    main_sess = requests.Session()
    main_sess.headers.update({"User-Agent": "Mozilla/5.0"})

    # Setopati
    for cat, tmpl in _SETO_CATS:
        empty = 0
        for page in range(1, 101):
            url = tmpl.format(page=page)
            try:
                r = main_sess.get(url, timeout=12, verify=False)
                if r.status_code != 200: continue
                soup = BeautifulSoup(r.text, "html.parser")
                links = {a["href"] for a in soup.find_all("a", href=True) if re.search(r"/\d+$", a["href"])}
                new_links = {("https://www.setopati.com" + u if u.startswith("/") else u).split("?")[0] for u in links if "/social/" in u or "/politics/" in u or "/opinion/" in u}
                new_links = {u for u in new_links if u not in seto_seen}
                if not new_links:
                    empty += 1
                    if empty >= 5: break
                    continue
                empty = 0
                for u in new_links:
                    seto_seen.add(u)
                    dl_q.put(("seto", u))
                dl_q.join()
                with lock:
                    seto_seen_path.write_text("\n".join(seto_seen), encoding="utf-8")
                    print(f"Setopati: {cat} page {page} | saved={saved['seto']:,}")
            except: pass

    # Ratopati
    for cat, tmpl in _RATO_CATS:
        empty = 0
        for page in range(1, 101):
            url = tmpl.format(page=page)
            try:
                r = main_sess.get(url, timeout=12, verify=False)
                if r.status_code != 200: continue
                soup = BeautifulSoup(r.text, "html.parser")
                links = {"https://www.ratopati.com" + a["href"] if a["href"].startswith("/") else a["href"] for a in soup.find_all("a", href=True) if "/story/" in a["href"]}
                new_links = {u.split("?")[0] for u in links if u not in rato_seen}
                if not new_links:
                    empty += 1
                    if empty >= 5: break
                    continue
                empty = 0
                for u in new_links:
                    rato_seen.add(u)
                    dl_q.put(("rato", u))
                dl_q.join()
                with lock:
                    rato_seen_path.write_text("\n".join(rato_seen), encoding="utf-8")
                    print(f"Ratopati: {cat} page {page} | saved={saved['rato']:,}")
            except: pass

    for _ in range(n_threads): dl_q.put(None)
    for w in workers: w.join()
    print(f"Setopati & Ratopati done: SETO={saved['seto']:,}, RATO={saved['rato']:,}")

run_setopati_ratopati()

In [ ]:
all_files = sorted(RAW_MANUAL_DIR.glob("*.jsonl"))
print(f"{'File':<50}  {'Lines':>8}  {'Words':>12}  {'Chars':>14}")
print("-" * 90)
grand_lines = grand_words = grand_chars = 0
for fpath in all_files:
    lines = words = chars = 0
    with open(fpath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try: text = json.loads(line).get("text", "")
            except: text = line
            lines += 1; words += len(text.split()); chars += len(text)
    grand_lines += lines; grand_words += words; grand_chars += chars
    print(f"{fpath.name:<50}  {lines:>8,}  {words:>12,}  {chars:>14,}")
print("-" * 90)
print(f"{'TOTAL':<50}  {grand_lines:>8,}  {grand_words:>12,}  {grand_chars:>14,}")
print(f"\nTotal: {grand_words/1e6:.1f}M words")